# TSFM H2S backtests on a Colab GPU

Runs `scripts/run_backtest.py` for the heavier configurations (Timestral-123M/209M,
TimesFM 3.0, long contexts, extended winter origins) on a GPU, then saves
`results/*.parquet` to Google Drive so `notebooks/04_leaderboard.py` can read
them locally.

**Runtime → Change runtime type → GPU (T4 is enough).**

The repo is private. Add a fine-grained GitHub token with read access to
`TheResilientCollective/tj_h2s_foundation_models` as a Colab secret named
`GH_TOKEN` (key icon in the left sidebar).

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/TheResilientCollective/tj_h2s_foundation_models.git
%cd tj_h2s_foundation_models
!pip install -q -e ".[timestral,timesfm]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Choose runs

Each tuple is `(model, mode, context, origins)`. `production` = the 711
validation-store origins (paired vs XGBoost); `extended` = every 6 h from
2025-10-01 (covers the winter event season).

In [ ]:
RUNS = [
    ("timesfm",        "univariate",              2048, "production"),
    ("timesfm",        "covariates",              2048, "production"),
    ("timesfm",        "multivariate_covariates", 2048, "production"),
    ("timestral-209M", "univariate",              2048, "production"),
    ("timestral-209M", "covariates",              2048, "production"),
    ("timestral-209M", "multivariate_covariates", 2048, "production"),
    ("timestral-28M",  "covariates",              8192, "production"),
    # winter events, no XGBoost pairing:
    ("timestral-209M", "covariates",              2048, "extended"),
    ("timesfm",        "covariates",              2048, "extended"),
    ("climatology",    "univariate",              2048, "extended"),
    ("persistence",    "univariate",              2048, "extended"),
]

In [ ]:
import subprocess, time
for model, mode, ctx, origins in RUNS:
    t = time.time()
    batch = 16 if model == "timesfm" else 64
    cmd = ["python", "scripts/run_backtest.py", "--model", model, "--mode", mode, "--context", str(ctx),
           "--origins", origins, "--device", "cuda", "--batch-size", str(batch)]
    print(" ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-1] if r.returncode == 0 else r.stderr[-2000:], f"({time.time()-t:.0f}s)")

## Save results to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/tsfm_h2s_results
!cp results/*.parquet /content/drive/MyDrive/tsfm_h2s_results/
!ls -la /content/drive/MyDrive/tsfm_h2s_results

## Quick look (same scoring as `04_leaderboard`)

In [ ]:
from tsfm_h2s import compare, metrics
res = compare.load_results("production")
paired = compare.paired(res)
cols = ["label", "product", "n", "mae", "spearman", "mean_cost", "n_ge10", "recall10", "auc10", "brier10", "n_ge30", "auc30"]
metrics.summarize(paired, by=["label", "product"])[cols].round(3).sort_values(["product", "mean_cost"])